In [1]:
import requests
import json
import base64
import time
import os
import pandas as pd
from IPython.display import display, Markdown

# ---------- Feature detection (same as before) ----------
def get_model_features(model_name):
    csv_path = r"E:\llm\Model\csv\all_models_combined.csv"
    input_type = None
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            row = df[df['Name'] == model_name]
            if not row.empty:
                input_type = row.iloc[0].get('Input', '')
        except:
            pass

    if input_type is not None:
        has_vision = 'Image' in str(input_type)
    else:
        low = model_name.lower()
        vision_patterns = ['vl', 'vision', 'llava', 'llama3.2-vision', 'llama4',
                           'gemma3', 'gemma4', 'ministral', 'mistral-small3.1',
                           'mistral-medium-3.5', 'medgemma', 'qwen2.5vl', 'qwen3-vl']
        has_vision = any(p in low for p in vision_patterns)

    low = model_name.lower()
    thinking_patterns = ['thinking', 'reasoning', 'deepseek-r1', 'phi4-reasoning',
                         'phi4-mini-reasoning', 'nemotron-3.5-lightning',
                         'nemotron-3-nano', 'nemotron-3-super']
    has_thinking = any(p in low for p in thinking_patterns)
    if 'deepseek-r1' in low:
        has_thinking = True

    return has_thinking, has_vision

# ---------- Image helpers ----------
def image_to_base64_from_path(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def image_to_base64_from_url(image_url):
    resp = requests.get(image_url, timeout=30)
    resp.raise_for_status()
    return base64.b64encode(resp.content).decode("utf-8")

def image_to_base64(image_input):
    if image_input.startswith(("http://", "https://")):
        return image_to_base64_from_url(image_input)
    else:
        return image_to_base64_from_path(image_input)

# ---------- Main function ----------
def chat_aware(prompt, model="qwen3:4b", image_input=None):
    has_thinking, has_vision = get_model_features(model)

    if has_vision and image_input is None:
        print(f"🔍 Model '{model}' supports images.")
        img = input("Enter image path or URL (or press Enter to skip): ").strip()
        if img:
            image_input = img

    message = {"role": "user", "content": prompt}
    if image_input:
        try:
            message["images"] = [image_to_base64(image_input)]
        except Exception as e:
            print(f"❌ Error loading image: {e}")
            return

    payload = {
        "model": model,
        "messages": [message],
        "stream": True,
    }

    resp = requests.post(
        "http://localhost:11434/api/chat",
        json=payload,
        stream=True,
        timeout=300,
    )
    resp.raise_for_status()

    start_time = time.time()
    first_thinking_time = None
    first_content_time = None
    thinking_text = ""
    content_text = ""

    handle = display(Markdown(""), display_id=True)

    features = []
    if has_thinking:
        features.append("🧠 Thinking")
    if has_vision:
        features.append("👁️ Vision")
    features_str = ", ".join(features) if features else "None"
    header = f"### Model: {model}\n**Features:** {features_str}\n\n"

    for line in resp.iter_lines(decode_unicode=True):
        if not line:
            continue
        data = json.loads(line)
        msg = data.get("message", {})

        if "thinking" in msg and msg["thinking"]:
            thinking_text += msg["thinking"]
            if first_thinking_time is None:
                first_thinking_time = time.time()

        if "content" in msg and msg["content"]:
            content_text += msg["content"]
            if first_content_time is None:
                first_content_time = time.time()

        md = header
        if thinking_text:
            md += f"## 🤔 Thinking\n\n{thinking_text}\n\n---\n\n"
        md += f"## 💬 Answer\n\n{content_text}"

        handle.update(Markdown(md))

        if data.get("done"):
            break

    total_time = time.time() - start_time

    timings = ""
    if first_thinking_time:
        timings += f"**TTFT (thinking):** {first_thinking_time - start_time:.3f}s  \n"
    if first_content_time:
        timings += f"**TTFT (content):** {first_content_time - start_time:.3f}s  \n"
    timings += f"**Total time:** {total_time:.3f}s"

    handle.update(Markdown(md + "\n---\n" + timings))

    return thinking_text, content_text

In [5]:
# List all installed models (from Ollama)
import requests
installed = [m['name'] for m in requests.get("http://localhost:11434/api/tags").json()['models']]
print("Installed models:")
for i, m in enumerate(installed, 1):
    print(f"{i}. {m}")

# Choose model by typing its name
model_name = input("Enter model name (exact from list): ")

# Call chat
prompt = "Explain the difference between a research hypothesis, a mathematical model, and a simulation model."
chat_aware(prompt, model=model_name)

Installed models:
1. gemma3:270m-it-qat
2. qwen2.5-coder:3b
3. qwen3:4b
4. qwen3:1.7b
5. qwen3:0.6b
6. qwen2.5-coder:1.5b-base-q4_K_M
7. qwen2.5-coder:0.5b-base-q5_0
8. qwen2.5-coder:0.5b-base-q4_K_M
9. qwen2.5-coder:1.5b-instruct-q4_K_M
10. qwen2.5-coder:1.5b-base-q4_K_S
11. qwen2.5-coder:1.5b-base-q4_1
12. qwen2.5-coder:1.5b-base-q4_0
13. qwen2.5-coder:1.5b-base-q3_K_L
14. qwen2.5-coder:1.5b-base-q3_K_M
15. qwen2.5-coder:1.5b-base-q3_K_S
16. qwen2.5-coder:1.5b-base-q2_K
17. qwen2.5-coder:1.5b-base
18. qwen2.5-coder:0.5b-instruct-fp16
19. qwen2.5-coder:0.5b-instruct-q8_0
20. qwen2.5-coder:0.5b-instruct-q6_K
21. qwen2.5-coder:0.5b-instruct-q5_K_M
22. qwen2.5-coder:0.5b-instruct-q5_K_S
23. qwen2.5-coder:0.5b-instruct-q5_1
24. qwen2.5-coder:0.5b-instruct-q5_0
25. qwen2.5-coder:0.5b-instruct-q4_K_M
26. qwen2.5-coder:0.5b-instruct-q4_K_S
27. qwen2.5-coder:0.5b-instruct-q4_1
28. qwen2.5-coder:0.5b-instruct-q4_0
29. qwen2.5-coder:0.5b-instruct-q3_K_L
30. qwen2.5-coder:0.5b-instruct-q3_K_M
31

### Model: gemma4:e2b-it-qat
**Features:** 👁️ Vision

## 🤔 Thinking

Here's a thinking process to structure the explanation of the three models:

1.  **Analyze the Request:** The user wants an explanation of the difference between three specific concepts:
    *   Research Hypothesis
    *   Mathematical Model
    *   Simulation Model

2.  **Define the Core Purpose of Each Concept (Mental Outline):**

    *   **Research Hypothesis:** This is the *starting point*. It's a testable, educated guess about a relationship between variables. It's about *theory* and *prediction*. (Often stated simply as an $X$ affects $Y$).
    *   **Mathematical Model:** This is the *tool* or *language*. It's a set of equations, variables, and logical relationships designed to capture the essence of the real-world system, allowing for analytical prediction and inference. It's about *structure* and *analysis*.
    *   **Simulation Model:** This is the *execution*. It's using the mathematical model (or a simplified version of it) to run experiments and observe how the system behaves under various conditions. It's about *behavior* and *demonstration*.

3.  **Develop Detailed Explanations for Each:**

    *   *Hypothesis:* Needs to emphasize testability and the "if-then" structure. Mention it precedes the formal models.
    *   *Mathematical Model:* Needs to emphasize abstraction, variables, equations, and its role in prediction/inference.
    *   *Simulation Model:* Needs to emphasize dynamics, time steps, input/output, and its role in testing the hypothesis or exploring scenarios.

4.  **Establish Relationships and Interdependence (Crucial Step):** How do these three concepts interact? They form a logical progression.
    *   Hypothesis $\rightarrow$ leads to $\rightarrow$ Mathematical Model $\rightarrow$ is tested by $\rightarrow$ Simulation Model.

5.  **Create an Analogy/Example (to make it concrete):** A good real-world example helps illustrate the progression.
    *   *Example Theme:* A simple physical system (e.g., fluid dynamics, population growth, or chemical reactions).

6.  **Review and Refine (Self-Correction/Tone Check):** Ensure the explanation is clear, distinct, and uses precise terminology. (The provided image of a small physical object is irrelevant to the conceptual question, so I will focus solely on the definitions.)

7.  **Final Polish (Structuring the Output):** Use clear headings and bullet points for readability. (This results in the final structured answer.)

---

## 💬 Answer

This is a common set of concepts in science, engineering, and data analysis, and they represent distinct stages in the process of turning a real-world observation into a testable theory.

Here is a detailed explanation of the difference between a research hypothesis, a mathematical model, and a simulation model, along with how they relate to each other.

---

## 1. The Research Hypothesis (The Theory/The Guess)

The research hypothesis is the **starting point**. It is an educated guess or a tentative statement about the relationship between two or more variables in the real world.

### What it is:
*   **A Prediction:** It is a specific, testable statement that predicts how one factor (the independent variable) will affect another factor (the dependent variable).
*   **Testable:** A good hypothesis must be phrased in a way that it can be verified or refuted through experimentation or observation.
*   **Simple Statement:** It is a high-level concept (e.g., "As temperature increases, the reaction rate will increase.")

### Key Characteristics:
*   **Abstract:** It is often stated conceptually before any complex formulas are introduced.
*   **Directional:** It proposes a specific direction of change (e.g., "A causes B").
*   **Goal:** To direct the investigation and determine if the initial idea is supported by data.

**Analogy:** Imagine you notice that plants grow faster when watered. Your hypothesis would be: **"If I water a plant more frequently, then it will grow faster."**

---

## 2. The Mathematical Model (The Language/The Structure)

The mathematical model is the **framework or language** used to describe the real-world phenomenon. It is a set of equations, variables, and logical relationships that capture the essential structure and rules of the system.

### What it is:
*   **Abstraction:** It takes the messy, complex reality and strips it down into a simplified, quantifiable structure.
*   **Equations:** It uses mathematical notation (variables, constants, functions, operators) to define how the variables interact.
*   **Analytical Tool:** It allows scientists to perform calculations and make predictions that are impossible to make intuitively from the raw data alone.

### Key Characteristics:
*   **Structural:** It defines *how* the system works (the rules).
*   **Rigorous:** It must be logically consistent and solvable using mathematical principles.
*   **Goal:** To allow for precise, analytical prediction and inference *before* the system is physically built or run.

**Analogy:** To model the plant growth example above, a mathematical model would look something like this:
$$\text{Growth Rate} = k \times (\text{Water Amount})^n$$
*(Where $k$ is a constant and $n$ is an exponent describing the relationship between water and growth.)*

---

## 3. The Simulation Model (The Experiment/The Execution)

The simulation model is the **practical application** of the mathematical model. It is the process of "running" the equations to see how the system behaves under specific conditions.

### What it is:
*   **Execution:** It is the actual running of the mathematical model, often on a computer, over time.
*   **Dynamic Behavior:** It shows the *dynamic* results—how the system changes moment by moment as inputs change.
*   **Scenario Testing:** It allows scientists to test "what if" scenarios (e.g., "What happens if the temperature is increased by 20 degrees?").

### Key Characteristics:
*   **Visual:** Often results in graphs, charts, and videos that show the change over time.
*   **Dynamic:** It models motion, time dependence, and continuous processes.
*   **Goal:** To see if the mathematical predictions (derived from the model) align with what is actually observed or predicted in a controlled environment.

**Analogy:** If the mathematical model defines the formula for growth, the simulation model would be the computer program that takes those formulas and runs them 1,000 times, showing you a visual graph of how the plant grows over a month when given different amounts of water.

---

## Summary Table: Key Differences

| Feature | Research Hypothesis | Mathematical Model | Simulation Model |
| :--- | :--- | :--- | :--- |
| **Purpose** | To make a testable prediction. | To describe the system's structure and rules. | To execute the rules and observe behavior. |
| **Format** | A simple statement (If... then...). | Equations, variables, and functions. | A programmed process or computer run. |
| **Nature** | Conceptual and directional. | Abstract and analytical. | Dynamic and visual. |
| **Role** | The **Question** | The **Blueprint** (The "How"). | The **Experiment** (The "What happens?"). |
| **Output** | A prediction. | Calculated values/Relationships. | Graphs, trajectories, and data patterns. |
---
**TTFT (thinking):** 0.000s  
**TTFT (content):** 73.632s  
**Total time:** 210.947s

('Here\'s a thinking process to structure the explanation of the three models:\n\n1.  **Analyze the Request:** The user wants an explanation of the difference between three specific concepts:\n    *   Research Hypothesis\n    *   Mathematical Model\n    *   Simulation Model\n\n2.  **Define the Core Purpose of Each Concept (Mental Outline):**\n\n    *   **Research Hypothesis:** This is the *starting point*. It\'s a testable, educated guess about a relationship between variables. It\'s about *theory* and *prediction*. (Often stated simply as an $X$ affects $Y$).\n    *   **Mathematical Model:** This is the *tool* or *language*. It\'s a set of equations, variables, and logical relationships designed to capture the essence of the real-world system, allowing for analytical prediction and inference. It\'s about *structure* and *analysis*.\n    *   **Simulation Model:** This is the *execution*. It\'s using the mathematical model (or a simplified version of it) to run experiments and observe h